In [ ]:
import mlflow
import mlflow.lightgbm
import pandas as pd
import numpy as np
from pathlib import Path
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, precision_recall_curve
import joblib

DATA_DIR = Path.home() / "kkbox-churn" / "data" / "parquet"
MODEL_DIR = Path.home() / "kkbox-churn" / "models"

# SQLite backend
mlflow_dir = Path.home() / "kkbox-churn" / "mlflow"
mlflow_dir.mkdir(exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{mlflow_dir / 'mlflow.db'}")
mlflow.set_experiment("kkbox-churn-prediction")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow version: {mlflow.__version__}")

2026/06/07 17:53:03 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/07 17:53:03 INFO mlflow.store.db.utils: Updating database tables
2026/06/07 17:53:03 INFO mlflow.tracking.fluent: Experiment with name 'kkbox-churn-prediction' does not exist. Creating a new experiment.


MLflow tracking URI: sqlite:////Users/harshithnr/kkbox-churn/mlflow/mlflow.db
MLflow version: 3.13.0


In [ ]:
master = pd.read_parquet(DATA_DIR / "master_dataset.parquet")

drop_cols = [
    'msno', 'last_transaction_date', 'last_expire_date',
    'last_listen_date', 'first_listen_date',
    'cluster_k5', 'segment', 'churn_prob', 'monthly_revenue',
    'churn_prob_clipped', 'clv', 'retention_priority'
]
master = master.drop(columns=[c for c in drop_cols if c in master.columns])

for col in ['city', 'registered_via', 'bd_bucket', 'auto_renew_switch']:
    master[col] = master[col].astype('category')

master_raw = pd.read_parquet(DATA_DIR / "master_dataset.parquet")
apr_idx = master_raw[master_raw['last_expire_date'].dt.to_period('M') == '2017-04'].index
may_idx = master_raw[master_raw['last_expire_date'].dt.to_period('M') == '2017-05'].index

X = master.drop(columns=['is_churn'])
y = master['is_churn']

X_train, y_train = X.loc[apr_idx], y.loc[apr_idx]
X_val,   y_val   = X.loc[may_idx], y.loc[may_idx]

print(f"Train: {X_train.shape}, churn rate: {y_train.mean():.4f}")
print(f"Val:   {X_val.shape},   churn rate: {y_val.mean():.4f}")

Train: (800236, 101), churn rate: 0.0165
Val:   (79942, 101),   churn rate: 0.0502


In [ ]:
best_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'learning_rate': 0.05386267542952244,
    'num_leaves': 39,
    'min_child_samples': 159,
    'feature_fraction': 0.8526646900411762,
    'bagging_fraction': 0.8828494997505784,
    'bagging_freq': 1,
    'reg_alpha': 0.10229096094071723,
    'reg_lambda': 1.1827367316767054e-05,
    'min_split_gain': 0.7253877757449226,
    'is_unbalance': True,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

with mlflow.start_run(run_name="final_model_optuna_tuned"):
    # Log params
    mlflow.log_params(best_params)
    mlflow.log_param("train_cohort", "2017-04")
    mlflow.log_param("val_cohort", "2017-05")
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("val_size", len(X_val))

    # Train
    model = lgb.LGBMClassifier(**best_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=-1)
        ]
    )

    # Metrics
    val_preds = model.predict_proba(X_val)[:, 1]
    roc = roc_auc_score(y_val, val_preds)
    pr  = average_precision_score(y_val, val_preds)

    precisions, recalls, thresholds = precision_recall_curve(y_val, val_preds)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best_thresh = thresholds[f1_scores[:-1].argmax()]
    preds_binary = (val_preds >= best_thresh).astype(int)

    from sklearn.metrics import f1_score, precision_score, recall_score
    f1  = f1_score(y_val, preds_binary)
    prec = precision_score(y_val, preds_binary)
    rec  = recall_score(y_val, preds_binary)

    mlflow.log_metric("roc_auc", roc)
    mlflow.log_metric("pr_auc", pr)
    mlflow.log_metric("f1_churn", f1)
    mlflow.log_metric("precision_churn", prec)
    mlflow.log_metric("recall_churn", rec)
    mlflow.log_metric("best_iteration", model.best_iteration_)
    mlflow.log_metric("optimal_threshold", best_thresh)

    # Log model artifact
    mlflow.lightgbm.log_model(model, "lgbm_churn_final")

    # Log feature importance
    import pandas as pd
    feat_imp = pd.DataFrame({
        'feature': X_train.columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    feat_imp.to_csv("/tmp/feature_importance.csv", index=False)
    mlflow.log_artifact("/tmp/feature_importance.csv")

    # Log SHAP plot
    mlflow.log_artifact(str(MODEL_DIR / "shap_summary.png"))
    mlflow.log_artifact(str(MODEL_DIR / "shap_waterfall_highrisk.png"))
    mlflow.log_artifact(str(MODEL_DIR / "shap_waterfall_lowrisk.png"))

    print(f"ROC-AUC: {roc:.4f}")
    print(f"PR-AUC:  {pr:.4f}")
    print(f"F1:      {f1:.4f}")
    print(f"Run logged successfully.")

2026/06/07 17:54:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 17:54:15 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


ROC-AUC: 0.9481
PR-AUC:  0.4287
F1:      0.4759
Run logged successfully.


In [ ]:
baseline_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'is_unbalance': True,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

with mlflow.start_run(run_name="baseline_default_params"):
    mlflow.log_params(baseline_params)
    mlflow.log_param("train_cohort", "2017-04")
    mlflow.log_param("val_cohort", "2017-05")
    mlflow.log_param("n_features", X_train.shape[1])

    model_base = lgb.LGBMClassifier(**baseline_params)
    model_base.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=-1)
        ]
    )

    val_preds_base = model_base.predict_proba(X_val)[:, 1]
    roc_base = roc_auc_score(y_val, val_preds_base)
    pr_base  = average_precision_score(y_val, val_preds_base)

    precisions, recalls, thresholds = precision_recall_curve(y_val, val_preds_base)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
    best_thresh = thresholds[f1_scores[:-1].argmax()]
    preds_binary = (val_preds_base >= best_thresh).astype(int)

    from sklearn.metrics import f1_score, precision_score, recall_score
    mlflow.log_metric("roc_auc", roc_base)
    mlflow.log_metric("pr_auc", pr_base)
    mlflow.log_metric("f1_churn", f1_score(y_val, preds_binary))
    mlflow.log_metric("precision_churn", precision_score(y_val, preds_binary))
    mlflow.log_metric("recall_churn", recall_score(y_val, preds_binary))
    mlflow.log_metric("best_iteration", model_base.best_iteration_)

    print(f"Baseline ROC-AUC: {roc_base:.4f}")
    print(f"Baseline PR-AUC:  {pr_base:.4f}")
    print("Baseline run logged.")

# Log failed experiments as separate runs with notes
with mlflow.start_run(run_name="experiment_scale_pos_weight"):
    mlflow.log_params({**baseline_params, 
                       'scale_pos_weight': 18.9,
                       'metric': 'average_precision'})
    mlflow.log_param("note", "worse_than_baseline")
    mlflow.log_metric("roc_auc", 0.8503)
    mlflow.log_metric("pr_auc", 0.3702)
    mlflow.log_metric("f1_churn", 0.43)
    print("scale_pos_weight experiment logged.")

with mlflow.start_run(run_name="experiment_velocity_features"):
    mlflow.log_params(best_params)
    mlflow.log_param("n_features", 114)
    mlflow.log_param("note", "velocity_features_added_worse")
    mlflow.log_metric("roc_auc", 0.9360)
    mlflow.log_metric("pr_auc", 0.4032)
    mlflow.log_metric("f1_churn", 0.45)
    print("Velocity features experiment logged.")

Baseline ROC-AUC: 0.9468
Baseline PR-AUC:  0.4174
Baseline run logged.
scale_pos_weight experiment logged.
Velocity features experiment logged.


In [ ]:
import optuna

# Log Optuna study summary
with mlflow.start_run(run_name="optuna_study_50_trials"):
    mlflow.log_param("n_trials", 50)
    mlflow.log_param("optimization_direction", "maximize")
    mlflow.log_param("optimization_metric", "roc_auc")
    mlflow.log_metric("best_auc", 0.9481)
    mlflow.log_metric("best_trial", 38)
    mlflow.log_params({
        'best_learning_rate': 0.05386267542952244,
        'best_num_leaves': 39,
        'best_min_child_samples': 159,
        'best_feature_fraction': 0.8526646900411762,
        'best_bagging_fraction': 0.8828494997505784,
        'best_reg_alpha': 0.10229096094071723,
        'best_reg_lambda': 1.1827367316767054e-05,
        'best_min_split_gain': 0.7253877757449226,
    })
    print("Optuna study logged.")

# Log segment model results
segment_results = {
    'Champion':       {'roc': 0.9599, 'pr': 0.4080, 'f1': 0.468, 'val_size': 61618},
    'Power_Listener': {'roc': 0.9531, 'pr': 0.4371, 'f1': 0.437, 'val_size': 12127},
    'At_Risk':        {'roc': 0.6447, 'pr': 0.3937, 'f1': 0.410, 'val_size': 4450},
    'New_Reengaged':  {'roc': 0.8482, 'pr': 0.3557, 'f1': 0.404, 'val_size': 1747},
}

for seg, res in segment_results.items():
    with mlflow.start_run(run_name=f"segment_model_{seg}"):
        mlflow.log_param("segment", seg)
        mlflow.log_param("val_size", res['val_size'])
        mlflow.log_param("approach", "segment_specific_lgbm")
        mlflow.log_metric("roc_auc", res['roc'])
        mlflow.log_metric("pr_auc", res['pr'])
        mlflow.log_metric("f1_churn", res['f1'])
        print(f"Segment {seg} logged: AUC {res['roc']:.4f}")

print("\nAll runs logged. Launch MLflow UI with:")
print(f"mlflow ui --backend-store-uri sqlite:///{Path.home() / 'kkbox-churn' / 'mlflow' / 'mlflow.db'}")

Optuna study logged.
Segment Champion logged: AUC 0.9599
Segment Power_Listener logged: AUC 0.9531
Segment At_Risk logged: AUC 0.6447
Segment New_Reengaged logged: AUC 0.8482

All runs logged. Launch MLflow UI with:
mlflow ui --backend-store-uri sqlite:////Users/harshithnr/kkbox-churn/mlflow/mlflow.db
